# Decentralized Reward Learning

Trains **N separate models** (one per agent).
- Uses corrected trajectory logic.
- Logs detailed errors per category per agent.
- No memory leaks.

In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 2
NUM_APPLES = 5

REWARD_SCHEME = "1_over_n"

MODEL_TYPE = "CNN"
MLP_HIDDEN_DIM = 32
MLP_NUM_LAYERS = 2
CNN_CONV_CHANNELS = [8]
CNN_KERNEL_SIZE = 1
CNN_HEAD_HIDDEN_DIM = 8
CNN_HEAD_NUM_LAYERS = 1

LR = 0.001
BATCH_SIZE = 64
DISCOUNT = 0.0
MAX_STEPS = 100000
EVAL_FREQ = 1

SOFT_THRESHOLD = 1.0
HARD_THRESHOLD = 1.0
NUM_TEST_SAMPLES = 1000

EXPERIMENT_NAME = "reward_decen"
OUTPUT_DIR = "outputs"
CSV_PATH = "outputs/experiment_results.csv"
SEED = 42

In [2]:
import sys
import time
import ast
from pathlib import Path
from typing import List, Dict, Callable

import numpy as np
import torch
from tqdm import tqdm

sys.path.append("../../")

from tadd_helpers.env_functions import State, init_empty_state
from tadd_helpers.setting_seed import set_all_seeds
from teleport_dynamic.base_value_model import BaseValueModelV2
from teleport_dynamic.models_decen_5ch import ValueCNNDecentralized5Ch, ValueMLPDecentralized5Ch
from teleport_dynamic.rewards_decentralized import (
    get_reward_minus1_2, get_reward_picker_1, get_reward_other_1, get_reward_1_over_n
)
from teleport_dynamic.orchard_generator import init_fixed_apples, teleport_agent
from teleport_dynamic.training_functions import train_reward_from_buffer

from teleport_dynamic.experiment_utils import (
    generate_decentralized_test_set,
    evaluate_decentralized_models,
    check_benchmark_decentralized,
    append_experiment_result,
    format_decen_eval_results_for_log,
    get_current_ram_mb
)
if isinstance(CNN_CONV_CHANNELS, str):
    CNN_CONV_CHANNELS = ast.literal_eval(CNN_CONV_CHANNELS)

REWARD_FUNCS = {
    "minus1_2": get_reward_minus1_2,
    "picker_1": get_reward_picker_1,
    "other_1": get_reward_other_1,
    "1_over_n": get_reward_1_over_n,
}
reward_func = REWARD_FUNCS[REWARD_SCHEME]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_all_seeds(SEED)

# Define Architecture String for Filename
if MODEL_TYPE == "CNN":
    # Format: conv[16,32]_k1_head32x1
    chan_str = str(CNN_CONV_CHANNELS).replace(" ", "")
    arch_str = f"conv{chan_str}_k{CNN_KERNEL_SIZE}_head{CNN_HEAD_HIDDEN_DIM}x{CNN_HEAD_NUM_LAYERS}"
else:
    # Format: mlp64x2
    arch_str = f"mlp{MLP_HIDDEN_DIM}x{MLP_NUM_LAYERS}"

# Construct Descriptive Log Filename
# Example: reward_decen_CNN_6x6_2ag_minus1_2_conv[16,32]_k1_head32x1.txt
log_filename = f"{EXPERIMENT_NAME}_{MODEL_TYPE}_{WIDTH}x{HEIGHT}_{NUM_AGENTS}ag_{REWARD_SCHEME}_{arch_str}.txt"

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUTPUT_DIR / log_filename

print(f"Logging to: {LOG_FILE}")

🌟 Setting all seeds to 42...
✅ Seeds and deterministic settings configured.
Logging to: outputs/reward_decen_CNN_6x6_2ag_minus1_2_conv[8]_k1_head32x1.txt


In [3]:
# Model Initialization
models: List[BaseValueModelV2] = []

for i in range(NUM_AGENTS):
    if MODEL_TYPE == "CNN":
        m = ValueCNNDecentralized5Ch(
            HEIGHT, WIDTH, i, LR, DISCOUNT,
            CNN_HEAD_HIDDEN_DIM, CNN_HEAD_NUM_LAYERS,
            CNN_CONV_CHANNELS, CNN_KERNEL_SIZE, DEVICE
        )
        cfg = f"conv={CNN_CONV_CHANNELS},k={CNN_KERNEL_SIZE},head={CNN_HEAD_HIDDEN_DIM}"
    else:
        m = ValueMLPDecentralized5Ch(
            HEIGHT, WIDTH, i, LR, DISCOUNT,
            MLP_HIDDEN_DIM, MLP_NUM_LAYERS, DEVICE
        )
        cfg = f"hidden={MLP_HIDDEN_DIM},layers={MLP_NUM_LAYERS}"
    models.append(m)

num_params = models[0].get_num_parameters()
print(f"Models: {NUM_AGENTS} x {num_params:,} params")

Models: 2 x 9,329 params


In [4]:
state = init_fixed_apples(WIDTH, HEIGHT, NUM_AGENTS, NUM_APPLES)

test_sets = generate_decentralized_test_set(
    HEIGHT, WIDTH, NUM_AGENTS, NUM_APPLES, NUM_TEST_SAMPLES, state.apples, reward_func, SEED + 1000
)
print("Test set generated.")

Test set generated.


In [5]:
with open(LOG_FILE, "w") as f:
    f.write(f"START Decentralized: {EXPERIMENT_NAME} | Scheme: {REWARD_SCHEME}\n")
    f.write(f"Grid: {WIDTH}x{HEIGHT}, Agents: {NUM_AGENTS}, Apples: {NUM_APPLES}\n")
    f.write(f"Params per model: {num_params:,} | Config: {cfg}\n")
    f.write(f"Benchmarks: Soft < {SOFT_THRESHOLD}%, Hard < {HARD_THRESHOLD}%\n")
    f.write(f"Log Format: Category: Mean Error % (avg across agents) / Max Error % (worst agent)\n")
    f.write("="*80 + "\n")

soft_benchmark_step = None
hard_benchmark_step = None
last_avg_loss = 0.0
start_time = time.time()

# Initialize c_t randomly before loop
c_t = np.random.randint(0, NUM_AGENTS)

print(f"Starting training for {MAX_STEPS} steps...")

for step in range(MAX_STEPS):
    rewards = reward_func(state, c_t)
    next_state = state.copy()
    teleport_agent(next_state, c_t)
    c_t_next = np.random.randint(0, NUM_AGENTS)
    
    step_losses = []
    for i in range(NUM_AGENTS):
        models[i].add_experience(state, next_state, rewards[i], c_t, c_t_next)
        l = train_reward_from_buffer(models[i], BATCH_SIZE)
        if l is not None: step_losses.append(l)
            
    if step_losses:
        last_avg_loss = float(np.mean(step_losses))
    
    state = next_state
    c_t = c_t_next
    
    if step > 0 and step % EVAL_FREQ == 0:
        results = evaluate_decentralized_models(models, test_sets)
        soft, hard = check_benchmark_decentralized(results, SOFT_THRESHOLD, HARD_THRESHOLD)
        ram_mb = get_current_ram_mb()
        
        if soft and not soft_benchmark_step: soft_benchmark_step = step
        if hard and not hard_benchmark_step: hard_benchmark_step = step
        
        status = "HARD" if hard else ("SOFT" if soft else "-")
        log_line = format_decen_eval_results_for_log(results, step, last_avg_loss, NUM_AGENTS)
        full_log = f"{log_line} | RAM: {ram_mb:.1f}MB | Bench: {status}"
        
        with open(LOG_FILE, "a") as f:
            f.write(full_log + "\n")
            
        if hard:
            print(f"Hard benchmark achieved at step {step}!")
            break

wall_time = time.time() - start_time

final_results = evaluate_decentralized_models(models, test_sets)
all_means = [r.mean_pct_error for cat in final_results.values() for r in cat.values()]
final_mean = max(all_means) if all_means else 0
all_maxes = [r.max_pct_error for cat in final_results.values() for r in cat.values()]
final_max = max(all_maxes) if all_maxes else 0

with open(LOG_FILE, "a") as f:
    f.write("="*80 + "\n")
    f.write(f"FINISHED. Time: {wall_time:.2f}s\n")
    f.write(f"Soft Step: {soft_benchmark_step}, Hard Step: {hard_benchmark_step}\n")
    f.write(f"Final Mean Err: {final_mean:.4f}%, Final Max Err: {final_max:.4f}%\n")
    f.write(format_decen_eval_results_for_log(final_results, step, last_avg_loss, NUM_AGENTS) + "\n")

print(f"Training done. Log saved to {LOG_FILE}")

Starting training for 100000 steps...


KeyboardInterrupt: 

In [ ]:
append_experiment_result(
    CSV_PATH, 
    "reward", 
    MODEL_TYPE, 
    False,  # decentralized
    f"{WIDTH}x{HEIGHT}", 
    NUM_AGENTS, 
    NUM_APPLES, 
    REWARD_SCHEME,
    cfg, 
    soft_benchmark_step, 
    hard_benchmark_step,
    step, 
    final_mean, 
    final_max, 
    wall_time, 
    # NEW:
    num_parameters=models[0].get_num_parameters(),
    kernel_size=CNN_KERNEL_SIZE if MODEL_TYPE == "CNN" else None,
    learning_method="reward",
    notes=""
)

In [ ]:
print("\n" + "="*80)
print("SANITY CHECK: Visualizing 3 samples per category")
print("="*80)

for category, cases in test_sets.items():
    if not cases: continue
    
    print(f"\n>>> CATEGORY: {category.upper()} <<<")
    samples = cases[:3]
    
    for i, case in enumerate(samples):
        print(f"\n--- Sample {i+1} ---")
        print(case.state)
        print(f"Acting Agent: {case.acting_agent_idx}")
        print(f"Self Agent (Perspective): {case.self_agent_idx}")
        
        # Get correct model for the self agent
        agent_model = models[case.self_agent_idx]
        
        # Visualize Input
        inp = agent_model.raw_state_to_nn_input(case.state, case.acting_agent_idx)
        print(f"NN Input Shape: {inp.shape}")
        if MODEL_TYPE == "CNN":
            # Channels: Apples, Others, Self, Self_Act, Other_Act
            print(f"  Ch0 (Apples)    Sum: {inp[0].sum()}")
            print(f"  Ch1 (Others)    Sum: {inp[1].sum()}")
            print(f"  Ch2 (Self)      Sum: {inp[2].sum()}")
            print(f"  Ch3 (Self_Act)  Sum: {inp[3].sum()}")
            print(f"  Ch4 (Other_Act) Sum: {inp[4].sum()}")

        pred = agent_model.get_value(case.state, case.acting_agent_idx)
        actual = case.true_reward
        print(f"PREDICTED: {pred:.4f} | ACTUAL: {actual:.4f} | ERROR: {abs(pred-actual):.4f}")


SANITY CHECK: Visualizing 3 samples per category

>>> CATEGORY: SELF_PICKER <<<

--- Sample 1 ---
--- Empty State (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (5, 5)
  Agent 1: (0, 4)

--- Agents (Count) ---
. . . . 1 .
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . 1

--- Apples (Count) ---
. . . . 1 .
. . . . . .
. . . 1 1 .
. 1 . . . .
. . 1 . . .
. . . . . .
Acting Agent: 1
Self Agent (Perspective): 1
NN Input Shape: (5, 6, 6)
  Ch0 (Apples)    Sum: 5.0
  Ch1 (Others)    Sum: 1.0
  Ch2 (Self)      Sum: 1.0
  Ch3 (Self_Act)  Sum: 1.0
  Ch4 (Other_Act) Sum: 0.0
PREDICTED: -1.0002 | ACTUAL: -1.0000 | ERROR: 0.0002

--- Sample 2 ---
--- Empty State (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (4, 2)
  Agent 1: (1, 5)

--- Agents (Count) ---
. . . . . .
. . . . . 1
. . . . . .
. . . . . .
. . 1 . . .
. . . . . .

--- Apples (Count) ---
. . . . 1 .
. . . . . .
. . . 1 1 .
. 1 . . . .
. . 1 . . .
. . . . . .
Acting Agent: 0
Self Agent (Perspective): 0
NN Input Sh